# Decision Tree Project — Telco Customer Churn
## Dataset Selection and Preprocessing

---

## 1. Mô tả Dataset

### Nguồn dữ liệu
- **Tên dataset:** IBM Telco Customer Churn
- **Nguồn:** IBM Sample Data Sets, công bố công khai trên Kaggle
- **Link:** https://www.kaggle.com/datasets/blastchar/telco-customer-churn
- **Tổ chức cung cấp:** IBM (International Business Machines Corporation)

### Thông tin cơ bản
| Thuộc tính | Giá trị |
|---|---|
| Số mẫu (samples) | 7.043 khách hàng |
| Số đặc trưng gốc (raw features) | 20 (sau khi loại customerID) |
| Số đặc trưng sau encode | 30 |
| Biến mục tiêu (target) | `Churn` — Yes/No |
| Loại bài toán | **Binary Classification** |

### Mô tả các đặc trưng chính
- **Thông tin khách hàng:** `gender`, `SeniorCitizen`, `Partner`, `Dependents`
- **Thông tin hợp đồng:** `tenure` (số tháng sử dụng), `Contract`, `PaperlessBilling`, `PaymentMethod`
- **Dịch vụ sử dụng:** `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`
- **Chi phí:** `MonthlyCharges`, `TotalCharges`
- **Biến mục tiêu:** `Churn` — khách hàng có rời bỏ dịch vụ hay không (Yes = rời bỏ, No = ở lại)

### Lý do dataset phù hợp với Decision Tree
1. **Hỗn hợp categorical và numerical:** Dataset có cả đặc trưng dạng phân loại (Yes/No, loại hợp đồng...) lẫn dạng số (tenure, MonthlyCharges). Decision Tree xử lý tốt cả hai loại sau khi encode.
2. **Dễ diễn giải kết quả:** Decision Tree tạo ra các quy tắc if-else có thể giải thích được (ví dụ: "Nếu hợp đồng Month-to-month VÀ tenure < 12 thì khả năng churn cao"). Điều này rất có giá trị trong bài toán kinh doanh như churn prediction.
3. **Không cần chuẩn hóa:** Decision Tree không yêu cầu scale dữ liệu, phù hợp với dataset có các đặc trưng có đơn vị rất khác nhau (tenure: 0–72, TotalCharges: 0–8000).
4. **Kích thước vừa phải:** 7.043 mẫu là đủ lớn để train/test, không quá nhỏ để underfit cũng không quá lớn để gây vấn đề tính toán.
5. **Bài toán phân loại nhị phân rõ ràng:** Nhãn `Churn` chỉ có 2 giá trị — hoàn toàn phù hợp với Decision Tree Classifier.

---
## 2. Import thư viện

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

---
## 3. Đọc và xem tổng quan dữ liệu

In [2]:
df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("=" * 50)
print(f"Shape ban dau: {df.shape}")
print(f"So mau: {df.shape[0]}")
print(f"So dac trung: {df.shape[1] - 1} (chua tinh cot customerID va Churn)")
print("=" * 50)
df.head()

Shape ban dau: (7043, 21)
So mau: 7043
So dac trung: 20 (chua tinh cot customerID va Churn)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Xem kiểu dữ liệu và thông tin tổng quan
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


---
## 4. Làm sạch dữ liệu (Data Cleaning)

**Vấn đề phát hiện:**
- Cột `TotalCharges` kiểu `object` thay vì `float` vì có 11 dòng chứa khoảng trắng `" "`
- Các cột object có thể có khoảng trắng thừa ở đầu/cuối chuỗi

In [4]:
# Bước 1: Xóa khoảng trắng thừa ở tất cả cột dạng string
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

In [5]:
# Bước 2: Chuyển TotalCharges sang kiểu số
# Các giá trị không hợp lệ (chuỗi rỗng sau strip) sẽ thành NaN
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Missing values sau khi convert TotalCharges:")
print(df.isnull().sum())

Missing values sau khi convert TotalCharges:
customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64


In [6]:
# Bước 3: Xóa 11 dòng có TotalCharges bị NaN
# Lý do: 11 dòng này là khách hàng mới (tenure = 0), TotalCharges chưa có giá trị
# → Chiếm ~0.16% tổng dữ liệu, an toàn để xóa
df = df.dropna()
print(f"Shape sau khi xóa NaN: {df.shape}")

Shape sau khi xóa NaN: (7032, 21)


In [ ]:
# Bước 4: Kiểm tra và xóa dòng trùng lặp
print(f"So dong trung lap: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Shape sau khi xu ly duplicate: {df.shape}")

In [7]:
# Bước 5: Xóa cột customerID (không có giá trị dự đoán)
df = df.drop(columns=["customerID"])
print("Da xoa cot customerID")
print(f"So dac trung con lai: {df.shape[1] - 1} features + 1 target")

Da xoa cot customerID
So dac trung con lai: 19 features + 1 target


---
## 5. Kiểm tra phân phối nhãn (Class Distribution)

Quan trọng với bài toán classification — cần biết dữ liệu có bị imbalanced không.

In [8]:
churn_counts = df["Churn"].value_counts()
churn_pct = df["Churn"].value_counts(normalize=True) * 100

print("Phan phoi nhan Churn:")
print(f"  No  (khong roi bo): {churn_counts['No']:>5} mau ({churn_pct['No']:.1f}%)")
print(f"  Yes (roi bo):       {churn_counts['Yes']:>5} mau ({churn_pct['Yes']:.1f}%)")
print()
print("Nhan xet: Du lieu bi mat can doi nhe (imbalanced ~73/27).")
print("Su dung stratify=y khi split de dam bao ti le nhan duoc giu nguyen.")

Phan phoi nhan Churn:
  No  (khong roi bo):  5163 mau (73.4%)
  Yes (roi bo):        1869 mau (26.6%)

Nhan xet: Du lieu bi mat can doi nhe (imbalanced ~73/27).
Su dung stratify=y khi split de dam bao ti le nhan duoc giu nguyen.


---
## 6. Encoding

**Phương pháp:** One-Hot Encoding với `pd.get_dummies(drop_first=True)`

- `drop_first=True` loại bỏ 1 cột trong mỗi nhóm để tránh multicollinearity (dummy variable trap)
- Ép kiểu về `int` (0/1) thay vì `bool` để tương thích tốt hơn với sklearn

In [9]:
df_encoded = pd.get_dummies(df, drop_first=True)

bool_cols = df_encoded.select_dtypes(include="bool").columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print(f"So dac trung sau encode: {df_encoded.shape[1] - 1} features + 1 target (Churn_Yes)")
print(f"Danh sach cot:")
print(df_encoded.columns.tolist())
df_encoded.head()

So dac trung sau encode: 30 features + 1 target (Churn_Yes)
Danh sach cot:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check', 'Churn_Yes']


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn_Yes
0,0,1,29.85,29.85,0,1,0,0,1,0,...,0,0,0,0,0,1,0,1,0,0
1,0,34,56.95,1889.50,1,0,0,1,0,0,...,0,0,0,1,0,0,0,0,1,0
2,0,2,53.85,108.15,1,0,0,1,0,0,...,0,0,0,0,0,1,0,0,1,1
3,0,45,42.30,1840.75,1,0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0
4,0,2,70.70,151.65,0,0,0,1,0,0,...,0,0,0,0,0,1,0,1,0,1


---
## 7. Tách X, y và Chia Train/Test

In [ ]:
X = df_encoded.drop(columns=["Churn_Yes"])
y = df_encoded["Churn_Yes"]

print(f"So mau: {X.shape[0]}")
print(f"So feature dung cho model: {X.shape[1]}")
print(f"Phan phoi y: 0 (No) = {(y==0).sum()}, 1 (Yes) = {(y==1).sum()}")

In [ ]:
# Chia 80% train, 20% test
# stratify=y: đảm bảo tỉ lệ churn được giữ nguyên trong cả 2 tập
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Ket qua chia train/test:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  y_train: {y_train.shape} | Churn rate: {y_train.mean()*100:.1f}%")
print(f"  y_test:  {y_test.shape}  | Churn rate: {y_test.mean()*100:.1f}%")

---
## 8. Lưu dữ liệu sạch (Bàn giao)

In [ ]:
df_encoded.to_csv("../data/clean/telco_clean.csv", index=False)
X_train.to_csv("../data/clean/X_train.csv", index=False)
X_test.to_csv("../data/clean/X_test.csv", index=False)
y_train.to_csv("../data/clean/y_train.csv", index=False)
y_test.to_csv("../data/clean/y_test.csv", index=False)

print("Da luu thanh cong cac file:")
print("  data/clean/telco_clean.csv  — Toan bo du lieu sach + encoded")
print("  data/clean/X_train.csv      — Tap huan luyen (features)")
print("  data/clean/X_test.csv       — Tap kiem tra (features)")
print("  data/clean/y_train.csv      — Nhan tap huan luyen")
print("  data/clean/y_test.csv       — Nhan tap kiem tra")

---
## 9. Tóm tắt kết quả

| Bước | Kết quả |
|---|---|
| Dataset gốc | 7.043 mẫu, 21 cột |
| Sau làm sạch | 7.032 mẫu (xóa 11 dòng TotalCharges=NaN) |
| Sau encoding | 31 cột (30 features + 1 target `Churn_Yes`) |
| Tập Train | 5.625 mẫu (80%) — Churn rate ≈ 26.5% |
| Tập Test | 1.407 mẫu (20%) — Churn rate ≈ 26.5% |
| Biến mục tiêu | `Churn_Yes`: 0 = Không rời bỏ, 1 = Rời bỏ |

**Dữ liệu sạch đã sẵn sàng để huấn luyện mô hình Decision Tree.**